In [1]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, TFAutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

In [2]:
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name, return_dict=True)
model.gradient_checkpointing_enable()

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print("Python")

2.6.0+cu124
True
NVIDIA GeForce RTX 4070 Ti SUPER
Python


In [4]:
input_text = "We went through many concepts and code in this article. We started with a brief description of the need for text summarization models and then moved on to using a pretrained model for summarization. Upon learning its limitations, we trained our own T5 model for summarization. Although on a small dataset, the post-training performance on the model was quite impressive. We did not stop there. Creating a Gradio app showed the possible applications that canbe built using text summarization models."

In [5]:
input_ids = tokenizer("Translate English to French: "+input_text, return_tensors='pt').input_ids
output_ids = model.generate(input_ids, max_length=512, num_beams=4, early_stopping=True)
output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(output)

Nous avons passé en revue de nombreux concepts et codes dans cet article. Nous avons commencé par une brève description de la nécessité de modèles de sommation de textes, puis nous avons procédé à l'utilisation d'un modèle de sommation et nous avons ensuite formé notre propre modèle T5 pour la sommation. Bien qu'il s'agisse d'un petit ensemble de données, le rendement après la formation du modèle était assez impressionnant.


In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="E:\Work\ML\Config Generation Server\datasets\draft_1.jsonl")
dataset = dataset["train"].train_test_split(test_size=0.1) 


In [ ]:
print(dataset.column_names)

{'train': ['input', 'output'], 'test': ['input', 'output']}


In [ ]:
print(dataset["test"][0])

In [9]:
print(tokenizer(['This is a test', 'Another example'], max_length=256, padding='max_length', truncation=True))


{'input_ids': [[100, 19, 3, 9, 794, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2351, 677, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [ ]:
import json
def preprocess(example):
    # print(f"Type of example['input']: {type(example['input'])}")
    # print(f"Type of first element in example['input']: {type(example['input'][0])}")
    # print(f"Example input[0]: {example['input'][0]}")
    
    # print(f"Type of example['output']: {type(example['output'])}")
    # print(f"Type of first element in example['output']: {type(example['output'][0])}")
    # print(f"Example output[0]: {example['output'][0]}")
    inputs = example["input"]
    # Convert dict outputs to JSON strings
    outputs = [json.dumps(out) if isinstance(out, dict) else out for out in example["output"]]
    model_inputs = tokenizer(
        inputs,
        max_length=256,
        padding="max_length",
        truncation=True
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            outputs,
            max_length=256,
            padding="max_length",
            truncation=True
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess, batched=True)


In [11]:
import transformers
print(transformers.__version__)
print(Seq2SeqTrainingArguments.__module__)

4.52.2
transformers.training_args_seq2seq


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="C:\\Users\\User\\Documents\\Work\\Config Generation Server\\Checkpoints_fine_tuned_t5\\t5-small-checkpoints",
    eval_strategy ="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="./logs",
    predict_with_generate=True,
    fp16=True,               
    save_strategy="epoch",    
    report_to="none",        
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
)

C:\Users\User\AppData\Local\Temp\ipykernel_20588\1581787282.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [15]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.105600,0.074937
2,0.086100,0.068178
3,0.077700,0.064618
4,0.071000,0.063177
5,0.069100,0.062433
6,0.067600,0.061911
7,0.066800,0.061420
8,0.065600,0.061290
9,0.064900,0.060954
10,0.064800,0.060907


TrainOutput(global_step=6360, training_loss=0.07277902717110496, metrics={'train_runtime': 1592.9652, 'train_samples_per_second': 127.68, 'train_steps_per_second': 3.993, 'total_flos': 1.376358450069504e+16, 'train_loss': 0.07277902717110496, 'epoch': 10.0})

In [ ]:
saved_model_path = "C:\\Users\\User\\Documents\\Work\\Config Generation Server\\Saved_fine_tuned_model\\t5-small-fine-tuned-saved-model"

In [ ]:

trainer.save_model(saved_model_path)  
tokenizer.save_pretrained(saved_model_path)  


In [ ]:
# fine_tuned_model_path = "./t5-fine_tuned_600"
fine_tuned_tokenizer = T5Tokenizer.from_pretrained(saved_model_path)
fine_tuned_model = T5ForConditionalGeneration.from_pretrained(saved_model_path)
def generate_output(input_text):
    input_ids = fine_tuned_tokenizer(input_text, return_tensors="pt").input_ids
    output_ids = fine_tuned_model.generate(
        input_ids,
        max_length=256,
        num_beams=4,
        early_stopping=True
    )
    return fine_tuned_tokenizer.decode(output_ids[0], skip_special_tokens=True)

#Test sample
# print(dataset["test"][0]["input"])
# print(dataset["test"][0]["output"])
# print(generate_output(dataset["test"][0]["input"]))
test_input = "Design a swiss-army knife config with edge shortcuts to email, music, messages, and compass, and circles for weather, battery, and step count. Occur weight learn security catch finally red."
print(generate_output(test_input))